# Metric Frequency Tracking from Long-Term Study Excel Files

## Project Background

Long Term studies are performed ahead of time to identify recurring issues that may affect real-time operations.

This project helps highlight areas where early mitigation, process improvement, or modeling improvements may reduce operational risk.


## Objective

The objective of this notebook is to read a group of Excel files, locate the `Metric` column in the first sheet of each file, and count how many times each metric appears across all files.

The final output provides a summary table showing:
- each metric category
- the total number of occurrences across the selected files




## Scope and Processing Rules

This notebook follows the rules below:

- Only Excel files in the current working folder are analyzed.
- Only the first sheet of each Excel file is read.
- The sheet name may vary from file to file.
- The code searches for a column named `Metric`.
- Fully blank rows are removed, but blank rows do not stop the read process.
- Metric values are standardized so that entries such as `ctg`, `Ctg`, and `CTG` are counted together.
- Files that do not contain a `Metric` column are skipped and reported.

## Step 1: Import Required Libraries

In [16]:
import pandas as pd
from pathlib import Path
from collections import Counter

## Step 2: Define the File Location

In [17]:
folder_path = Path(".")

## Step 3: Collect the Excel Files


In [18]:
excel_files = list(folder_path.glob("*.xlsx")) + list(folder_path.glob("*.xls"))

## Step 4: Read, Clean, and Count Metric Values

This is the main processing step of the notebook.

For each Excel file, the code:
- reads only the first sheet
- removes rows that are completely blank
- cleans the column names
- searches for the `Metric` column
- standardizes the text format in the `Metric` values
- counts each occurrence of each metric

If a file does not contain a `Metric` column, it is skipped and a message is printed.

In [19]:
metric_total_count = Counter()


for file in excel_files:
    try:
        df = pd.read_excel(file, sheet_name=0, dtype=str)
        df = df.dropna(how="all")
        df.columns = [str(col).strip() for col in df.columns]

        metric_col = None
        for col in df.columns:
            if col.lower() == "metric":
                metric_col = col
                break

        if metric_col is None:
            print(f"'Metric' column not found in first sheet of: {file.name}")
            continue

        metric_values = (
            df[metric_col]
            .dropna()
            .astype(str)
            .str.strip()
        )

        metric_values = metric_values[metric_values != ""]
        metric_values = metric_values.str.lower().str.title()

        metric_total_count.update(metric_values)

    except Exception as e:
        print(f"Error reading {file.name}: {e}")

'Metric' column not found in first sheet of: samin_table.xlsx


## Step 5: Create the Summary Table

This step converts the metric counts into a DataFrame and sorts the results from highest count to lowest count.

The displayed table provides a quick summary of the most frequent issue categories found in the long-term study files.


In [20]:
result_df = pd.DataFrame(
    metric_total_count.items(),
    columns=["Metric", "Total Count"]
).sort_values(by="Total Count", ascending=False)

from IPython.display import HTML, display
display(HTML(result_df.to_html(index=False)))

Metric,Total Count
Temporary Contingency,10
Ratings & Impedances,9
Modeling,5
Green Tag Risk Mitigation,4
Missed Or Incoorect Contingency,2
Powerflow,2
Alternate Switching Solution,2
Red Tag Risk Mitigation,2
Designation,2
Missing Orca Ticket,2


## Step 6: Export the Results

This step saves the final summary table to a new Excel file.

The exported file can be used for reporting, presentation, or additional analysis outside the notebook.


In [21]:
# Save result
output_folder = folder_path / "output"
output_folder.mkdir(exist_ok=True)

output_file = output_folder / "metric_total_count_first_sheet_only.xlsx"
result_df.to_excel(output_file, index=False)
print(f"Summary saved to: {output_file}")

Summary saved to: output/metric_total_count_first_sheet_only.xlsx


## Optional Validation / Debugging Cell

This optional section is useful for checking whether the files are being read correctly.

It helps confirm:
- which files are being processed
- the row and column counts before and after blank-row cleanup
- the detected column names
- whether the `Metric` column was found
- sample values from the `Metric` column

This section is mainly for troubleshooting and validation, and it is not required for the final output.


In [22]:
for file in excel_files:
    try:
        print(f"\nReading file: {file.name}")
        
        df = pd.read_excel(file, sheet_name=0, dtype=str)
        print("Rows, Columns before blank-row cleanup:", df.shape)

        df = df.dropna(how="all")
        print("Rows, Columns after blank-row cleanup:", df.shape)

        df.columns = [str(col).strip() for col in df.columns]
        print("Columns found:", list(df.columns))

        metric_col = None
        for col in df.columns:
            if col.lower() == "metric":
                metric_col = col
                break

        if metric_col is None:
            print(f"'Metric' column NOT found in: {file.name}")
            continue

        print(f"'Metric' column found in: {file.name}")

        metric_values = (
            df[metric_col]
            .dropna()
            .astype(str)
            .str.strip()
        )

        metric_values = metric_values[metric_values != ""]
        metric_values = metric_values.str.lower().str.title()

        print("Sample metric values:", metric_values.head(10).tolist())

        metric_total_count.update(metric_values)

    except Exception as e:
        print(f"Error reading {file.name}: {e}")



Reading file: file1.xlsx
Rows, Columns before blank-row cleanup: (58, 10)
Rows, Columns after blank-row cleanup: (53, 10)
Columns found: ['Request', 'Start', 'End', 'kV', 'Line', 'Station', 'Region', 'Comments', 'Metric', 'Details']
'Metric' column found in: file1.xlsx
Sample metric values: ['Ratings & Impedances', 'Green Tag Risk Mitigation', 'Temporary Contingency', 'Missed Or Incoorect Contingency', 'Modeling', 'Powerflow', 'Alternate Switching Solution', 'Temporary Contingency', 'Ratings & Impedances', 'Green Tag Risk Mitigation']

Reading file: file2.xlsx
Rows, Columns before blank-row cleanup: (58, 10)
Rows, Columns after blank-row cleanup: (53, 10)
Columns found: ['Request', 'Start', 'End', 'kV', 'Line', 'Station', 'Region', 'Comments', 'Metric', 'Details']
'Metric' column found in: file2.xlsx
Sample metric values: ['Ratings & Impedances', 'Green Tag Risk Mitigation', 'Temporary Contingency', 'Missed Or Incoorect Contingency', 'Modeling', 'Powerflow', 'Alternate Switching Solut